This is a shortened version of the Image Formation notebook.
It jumps straight to the full TEM image simulation, skipping the derivations.
Run the **Setup** cell first, then the interactive cells below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
import mrcfile as mrc
from scipy.interpolate import RegularGridInterpolator
from scipy.spatial.transform import Rotation
from scipy.stats import norm
from matplotlib.patches import Circle
%matplotlib widget

from functions import *
from functions import make_contrast_transfer_function as ctf

# Load protein density map
pdb = '8F76' # Human Olefactory receptor
pdb = '6RJH' # Apo-ferritin
m = mrc.open('{}.mrc'.format(pdb), 'r')
volume = np.asarray(m.data)
psize = m.voxel_size['x']

eV = 3e5   # 300 kV electron
i = 2      # default projection axis

# Grid coordinates for Fourier slice interpolation
grid = [np.arange(x) - x//2 for x in volume.shape]

pixels = volume.shape[0]
size = psize * pixels

def add_noise(arrayin, Total_counts):
    """Add Poisson counting noise to simulated data."""
    return np.random.poisson(arrayin * Total_counts)

def project_volume(vol, rot, tilt, psi, returnfft=False):
    R = Rotation.from_euler('zyz', [rot, tilt, psi], degrees=True).as_matrix()
    x = R[0].reshape((3,1)) * np.arange(-volume.shape[0]//2, volume.shape[0]//2).reshape((1, volume.shape[0]))
    y = R[1].reshape((3,1)) * np.arange(-volume.shape[1]//2, volume.shape[1]//2).reshape((1, volume.shape[0]))
    g = (x.reshape(3,volume.shape[0],1) + y.reshape(3,1,volume.shape[1])).reshape(3,np.prod(volume.shape[:2])).T
    fftprojection = vol(g).reshape(volume.shape[:2])
    projection = np.fft.ifftshift(np.real(np.fft.ifft2(np.fft.fftshift(fftprojection)))) / volume.shape[0]
    if returnfft:
        return projection, fftprojection
    else:
        return projection

Simulate a noisy, defocused TEM image of the protein in any orientation and compare it with the projected potential:

In [ ]:
fig,ax = plt.subplots(ncols=2, figsize=(8,4))
pixels = 256

# Fourier transform volume and set up slice interpolator
volfft = np.fft.ifftshift(np.fft.fftn(np.fft.fftshift(volume)))
vol = RegularGridInterpolator(grid, volfft, method="linear", bounds_error=False, fill_value=0)
pixels = volume.shape[0]

im_proj = ax[0].imshow(np.zeros((pixels,pixels)))
im = ax[1].imshow(np.zeros((pixels,pixels)), vmin=0.8, vmax=1.2)

for a,title in zip(ax,['Projected potential','TEM image']):
    a.set_axis_off()
    a.set_title(title)
fig.tight_layout()
plt.show()

def plot_propagator(rot,tilt,psi,dz,dose):
    """Function to plot the propagator"""
    proj = project_volume(vol,rot,tilt,psi,returnfft=False)
    wavefunction = np.exp(1j*proj)
    gaussian = Gaussian(1,[pixels,pixels],[size,size])
    propagator = ctf([pixels,pixels],[size,size],eV,df=dz*1e4,aberrations=cs(2.7))
    img = np.abs(np.fft.ifft2(np.fft.fft2(wavefunction)*propagator*np.fft.fft2(gaussian)))**2
    counts = dose*size*size
    img = img/np.sum(img)
    img = add_noise(img,counts)
    im_proj.set_data(proj)
    im_proj.set_clim(vmin=proj.min(),vmax=proj.max())
    im.set_data(img)
    im.set_clim(vmin=img.min(),vmax=img.max())

interact(plot_propagator,
         rot=widgets.FloatSlider(value=0.0, min=0, max=360, step=1, description='rot:'),
         tilt=widgets.FloatSlider(value=0.0, min=0, max=360, step=1, description='tilt:'),
         psi=widgets.FloatSlider(value=0.0, min=0, max=360, step=1, description='psi:'),
         dz=widgets.FloatSlider(value=0.0, min=-5, max=5, step=0.01,
                                description='$\\Delta$f ($\\mu$m):', readout_format='.2f'),
         dose=widgets.FloatLogSlider(value=40, base=10, min=1, max=5, step=0.01,
                                     description='e/Å$^2$:', readout_format='.2e'))

So how do we know what defocus was used in experiment? As well as an image of the particle we are simultaneously forming an image of the ice layer surrounding our particles, a Fourier transform of this image shows Thon rings which are mathematically related to our defocus and can be "fitted" by a computer program.

In [ ]:
from scipy.stats import norm
from matplotlib.patches import Circle

def ctfphase(eV,pixels,size,Cs,df):
    lambd = 1/wavev(eV)
    k = np.fft.fftfreq(pixels, d=size / pixels)[:pixels//2]
    return k, 2*np.pi*(df*lambd*k**2/2+Cs*lambd**3*k**4/4)

def ctf1d(eV,pixels,size,Cs,df):
    lambd = 1/wavev(eV)
    k = np.fft.fftfreq(pixels, d=size / pixels)[:pixels//2]
    return k, np.sin(2*np.pi*(df*lambd*k**2/2+Cs*lambd**3*k**4/4))

def ctf_zeros(eV,Cs,df,n=3):
    N = np.arange(1,n+1)
    lambd = 1/wavev(eV)
    a = df**2*lambd**2/4
    b = Cs*lambd**3/2
    L2 = df*lambd
    L1 = 2*np.sqrt(df**2*lambd**2/4+N*Cs*lambd**3/2)
    if L2>2*np.sqrt(df**2*lambd**2/4+Cs*lambd**3/2):
        num = L1 - L2
    else:
        N=-N
        num = -L2 - 2*np.sqrt(a+N*b)
    print(int(np.floor(np.abs(a/b))),a,b,N)
    denom = Cs*lambd**3
    return np.sqrt(num/denom)

# Make a Figure
pixels=512
fig,ax = plt.subplots(ncols=3,figsize=(9,3))
im = ax[0].imshow(np.zeros((512,512)),vmin=0.8,vmax=1.2)
im2 = ax[1].imshow(np.zeros((512,512)),vmin=0.8,vmax=1.2)
k,ctf_ = ctf1d(eV,pixels,size,2.7e7,-1e4)
lines, = ax[2].plot(k,ctf_,'k-')
lines2, = ax[2].plot(k,ctf_,'r-')
zlines = []
for i in range(3):
   zlines.append(ax[2].plot([],[],'r-')[0])
ax[2].set_xlim(0,k.max()/2)

for a,title in zip(ax[:2],['TEM image','Power spectrum']):
    a.set_axis_off()
    a.set_title(title)
ax[2].set_title('CTF')
fig.tight_layout()
fig.canvas.draw()
plt.show()

# Project protein
proj = np.mean(volume,axis=i)
particle = (1-0.1*proj/proj.max())*np.exp(1j*proj)
pixels = volume.shape[0]
size = psize*pixels
ice = norm(loc=5).rvs(size=[pixels,pixels])/5/2

def plot_defocused_image(dz,addice,addparticle):
    """Function to generate a phase contrast TEM image"""
    k,ctf_ = ctf1d(eV,pixels,size,2.7e7,dz*1e4)
    k,phase = ctfphase(eV,pixels,size,2.7e7,dz*1e4)
    lines.set_data(k,ctf_)
    lines2.set_data(k,phase)
    ax[2].set_ylim(phase.min(),phase.max())
    zeros = ctf_zeros(eV,2.7e7,dz*1e4)
    gaussian = Gaussian(1,[pixels,pixels],[size,size])
    w = np.ones([pixels,pixels])
    if addice:
        w = np.exp(1j*ice)*w
    if addparticle:
        w = particle*w
    propagator = ctf([pixels,pixels],[size,size],eV,df=dz*1e4,aberrations=cs(2.7))
    img = np.abs(np.fft.ifft2(np.fft.fft2(w)*propagator))**2
    img = np.real(np.fft.ifft2(np.fft.fft2(gaussian)*np.fft.fft2(img)))
    im.set_data(img)
    im.set_clim(vmin=img.min(),vmax=img.max())
    fft= np.abs(np.fft.fft2(img))
    fft[0,0]=0
    for i,ln in enumerate(zlines):
       ln.set_data([zeros[i],zeros[i]],[-1,1])
    im2.set_data(np.fft.fftshift(fft))
    im2.set_clim(vmin=fft.min(),vmax=fft.max())

interact(plot_defocused_image,
         dz=widgets.FloatSlider(value=0.0, min=-5, max=5, step=0.01,
                                description='$\\Delta$f ($\\mu$m):', readout_format='.2f'),
         addice=widgets.Checkbox(value=False,description='Add ice'),
         addparticle=widgets.Checkbox(value=True,description='Add particle'))

So how would we start to think about doing a cryo-EM reconstruction, that is taking many thousands of noisy, defocused images of our particle in different orientations and combining them all back into a 3D density map? Since applying a defocus can be modelled by a mathematical operation (multiplying by a contrast transfer function in Fourier space) the inverse mathematical operation (dividing by a contrast transfer function in Fourier space) can "restore" much of the information lost by defocus.

In [ ]:
from scipy.stats import norm

# Make a Figure
pixels = 256
fig,ax = plt.subplots(ncols=3)
im = ax[0].imshow(np.zeros((pixels,pixels)),vmin=0.8,vmax=1.2)
im2 = ax[1].imshow(np.zeros((pixels,pixels)),vmin=0.8,vmax=1.2)
im3 = ax[2].imshow(np.zeros((pixels,pixels)),vmin=0.8,vmax=1.2)

for a,title in zip(ax,['TEM image','Contrast transfer function (CTF)','CTF corrected image']):
    a.set_axis_off()
    a.set_title(title)
fig.tight_layout()
plt.show()

# Project protein
proj = crop(np.mean(volume,axis=i),[2*pixels,2*pixels])
wavefunction = (1-0.1*proj/proj.max())*np.exp(1j*proj)
pixels = volume.shape[0]
size = psize*pixels
ice = norm(loc=5).rvs(size=[pixels*2,pixels*2])/5/2

def plot_defocused_image(dz,addice):
    """Function to generate a phase contrast TEM image"""
    gaussian = Gaussian(1,[pixels*2,pixels*2],[size,size])
    if addice:
        w = np.exp(1j*ice)*wavefunction
    else:
        w = wavefunction
    propagator = ctf([2*pixels,2*pixels],[2*size,2*size],eV,df=dz*1e4,aberrations=cs(2.7))
    img = np.abs(np.fft.ifft2(np.fft.fft2(w)*propagator))**2
    img = np.real(np.fft.ifft2(np.fft.fft2(gaussian)*np.fft.fft2(img)))
    img = crop(img,[pixels,pixels])
    ctf_ = ctf([pixels,pixels],[size,size],eV,df=dz*1e4,aberrations=cs(2.7)).imag
    epsilon = 1e-1
    fftcor = np.real(np.fft.ifft2(np.fft.fft2(img)*ctf_/(epsilon+np.abs(ctf_)**2)))
    im.set_data(img)
    im.set_clim(vmin=img.min(),vmax=img.max())
    im2.set_data(np.fft.fftshift(ctf_))
    im2.set_clim(vmin=ctf_.min(),vmax=ctf_.max())
    im3.set_data(fftcor)
    im3.set_clim(vmin=fftcor.min(),vmax=fftcor.max())

interact(plot_defocused_image,
         dz=widgets.FloatSlider(value=0.0, min=-10, max=2, step=0.01,
                                description='$\\Delta$f ($\\mu$m):', readout_format='.2f'),
         addice=widgets.Checkbox(value=False,description='Add ice'))